# 🛰️ HỆ THỐNG TRUYỀN DỮ LIỆU QUA VỆ TINH IRIDIUM - ROCKBLOCK 9603
## 📡 MODULE GỬI (SENDER) - SERIAL / ID: 0235707 ➡️ TARGET: 0235708

### 📖 Nguyên lý định tuyến trực tiếp RockBLOCK-to-RockBLOCK (Direct Addressing):
* Theo chuẩn giao thức mạng vệ tinh của **Ground Control / Rock Seven**, để gửi trực tiếp từ một RockBLOCK tới một RockBLOCK khác:
  1. Tin nhắn ASCII nạp vào modem phát bắt buộc phải có tiền tố:
     $$\mathbf{AT+SBDWT=RB} + \mathbf{\langle Serial\_7\_ký\_tự\rangle} + \mathbf{\langle Nội\_dung\_tin\_nhắn\rangle}$$
  2. **Tiền tố:** Bắt đầu bằng 2 chữ cái `RB` + Số serial 7 chữ số của thiết bị nhận (ở đây là `0235708` ➡️ `RB0235708`).
  3. **Cơ chế Gateway:** Khi RockBLOCK 0235707 gọi `AT+SBDIX` đẩy gói tin lên chùm vệ tinh Iridium, trạm Gateway Rock Seven sẽ đọc tiền tố `RB0235708` để định tuyến tin nhắn đến đúng hộp thư của RockBLOCK 0235708.
  4. **Tự động bóc tách tiền tố:** Trước khi gói tin được đưa vào hàng đợi MT của RockBLOCK nhận 0235708, **Gateway sẽ tự động cắt bỏ 9 ký tự tiền tố `RB0235708`**, bên nhận khi đọc bằng `AT+SBDRT` sẽ nhận được đúng nội dung thuần túy!

--- 
### ⚙️ Các lệnh AT cốt lõi sử dụng trong Notebook này:
* `AT`: Kiểm tra modem phản hồi (OK).
* `AT+CSQ`: Kiểm tra mức sóng vệ tinh Iridium (0 đến 5 vạch).
* `AT+SBDD0`: Xóa bộ đệm MO (Mobile Originated Buffer) cũ.
* `AT+SBDWT=RB0235708<message>`: Ghi tin nhắn kèm địa chỉ nhận vào bộ đệm MO.
* `AT+SBDIX`: Khởi tạo phiên truyền dữ liệu vệ tinh SBD (Short Burst Data).


In [ ]:
# CELL 1: CẤU HÌNH KẾT NỐI VÀ CHỌN CỔNG COM
import serial
import serial.tools.list_ports
import time
from datetime import datetime

# ==============================================================================
# DANH SÁCH CỔNG SERIAL HIỆN TẠI TRÊN MÁY TÍNH
# ==============================================================================
print("🔍 Danh sách các cổng COM khả dụng:")
ports = list(serial.tools.list_ports.comports())
if not ports:
    print("   [!] Không tìm thấy cổng COM nào. Hãy kiểm tra kết nối cáp USB!")
for p in ports:
    print(f"   - {p.device}: {p.description}")

# ==============================================================================
# CẤU HÌNH THÔNG SỐ ROCKBLOCK SENDER VÀ TARGET RECEIVER
# ==============================================================================
# Thay đổi PORT theo cổng COM thực tế của bạn (ví dụ 'COM14', 'COM3')
PORT = "COM14"                   # <-- CỔNG COM CỦA ROCKBLOCK SENDER (0235707)
BAUDRATE = 19200                # Tốc độ baud mặc định của RockBLOCK 9603
SENDER_SERIAL = "0235707"       # Serial của module phát này
TARGET_RECEIVER_SERIAL = "0235708" # <-- SERIAL 7 CHỮ SỐ CỦA THIẾT BỊ NHẬN (RECEIVER)
LOG_FILE = f"rockblock_sender_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

print("\n--- THÔNG TIN CẤU HÌNH ---")
print(f"Cổng Serial      : {PORT} @ {BAUDRATE} bps")
print(f"Sender Serial    : {SENDER_SERIAL}")
print(f"Target Receiver  : {TARGET_RECEIVER_SERIAL} (Tiền tố: RB{int(TARGET_RECEIVER_SERIAL):07d})")
print(f"File lưu log     : {LOG_FILE}")


In [ ]:
# CELL 2: ĐỊNH NGHĨA CLASS ĐIỀU KHIỂN ROCKBLOCK SENDER
class RockBlockSender:
    """Lớp điều khiển module RockBLOCK gửi thông điệp SBD qua mạng vệ tinh Iridium."""
    
    MO_STATUS_DESC = {
        0: "Thành công: MO message đã được trạm vệ tinh Iridium tiếp nhận hoàn tất!",
        1: "Thành công: MO message đã gửi thành công, nhưng kích thước quá lớn đối với bên nhận.",
        2: "Thành công: MO message đã gửi thành công, nhưng trạm không xác định được vị trí vệ tinh.",
        32: "Thất bại: Không bắt được sóng vệ tinh Iridium hoặc tín hiệu quá yếu (Timeout).",
        33: "Thất bại: Mất kết nối vô tuyến trong quá trình truyền dữ liệu.",
        34: "Thất bại: Giao thức mạng vệ tinh báo bận / nghẽn kênh.",
        35: "Thất bại: Modem Iridium bị khóa hoặc SIM chưa đăng ký gói cước SBD."
    }

    def __init__(self, port, baudrate=19200, log_file=None):
        self.port = port
        self.baudrate = baudrate
        self.log_file = log_file
        self.ser = None
        self.imei = None

    def log(self, text):
        """In ra màn hình và ghi log kèm mốc thời gian."""
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{timestamp}] {text}"
        print(line)
        if self.log_file:
            with open(self.log_file, "a", encoding="utf-8") as f:
                f.write(line + "\n")

    def connect(self):
        """Mở cổng Serial kết nối với module RockBLOCK."""
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                bytesize=serial.EIGHTBITS,
                parity=serial.PARITY_NONE,
                stopbits=serial.STOPBITS_ONE,
                timeout=3
            )
            self.log(f"Đã mở thành công cổng {self.port} @ {self.baudrate} baud.")
            return True
        except Exception as e:
            self.log(f"[LỖI KẾT NỐI] Không thể mở cổng {self.port}: {e}")
            return False

    def disconnect(self):
        """Đóng kết nối Serial an toàn."""
        if self.ser and self.ser.is_open:
            self.ser.close()
            self.log(f"Đã đóng cổng {self.port}.")

    def send_cmd(self, cmd, wait=2):
        """Gửi lệnh AT, chờ và đọc chuỗi phản hồi từ modem."""
        if not self.ser or not self.ser.is_open:
            self.log("Cổng Serial chưa được mở!")
            return ""
        
        self.ser.reset_input_buffer()
        self.ser.write((cmd + "\r").encode("ascii"))
        time.sleep(wait)
        
        response = self.ser.read_all().decode("ascii", errors="ignore").strip()
        return response

    def init_modem(self):
        """Khởi tạo modem, tắt Echo (ATE0) và đọc số IMEI thiết bị."""
        self.log("--- Khởi tạo cấu hình Modem ---")
        # Kiểm tra lệnh AT cơ bản
        resp = self.send_cmd("AT", wait=1)
        if "OK" not in resp:
            self.log(f"Modem không phản hồi OK với lệnh AT. Phản hồi: {resp}")
            return False
        
        # Tắt echo lệnh (ATE0) và tắt điều khiển luồng phần cứng (AT&K0)
        self.send_cmd("ATE0", wait=1)
        self.send_cmd("AT&K0", wait=1)
        
        # Đọc số IMEI (AT+CGSN)
        imei_resp = self.send_cmd("AT+CGSN", wait=1)
        for line in imei_resp.splitlines():
            line = line.strip()
            if line.isdigit() and len(line) >= 14:
                self.imei = line
                break
        
        # Đọc phiên bản firmware (AT+CGMR)
        fw = self.send_cmd("AT+CGMR", wait=1)
        
        self.log(f"🔢 IMEI Sender   : {self.imei if self.imei else 'Không rõ'}")
        self.log(f"💾 Firmware      : {fw.replace('OK', '').strip()}")
        self.log("✅ Modem đã sẵn sàng hoạt động!")
        return True

    def check_signal(self):
        """Kiểm tra mức sóng vệ tinh Iridium (AT+CSQ). Trả về số nguyên từ 0 đến 5."""
        resp = self.send_cmd("AT+CSQ", wait=2)
        if "+CSQ:" in resp:
            try:
                bars_str = resp.split("+CSQ:")[1].strip().split()[0]
                bars = int(bars_str)
                bar_display = "■" * bars + "□" * (5 - bars)
                self.log(f"📶 Mức tín hiệu vệ tinh: [{bar_display}] ({bars}/5 vạch)")
                return bars
            except Exception:
                pass
        self.log(f"⚠️ Không đọc được mức sóng. Phản hồi: {resp}")
        return 0

    def wait_for_signal(self, min_bars=2, max_attempts=10, interval=5):
        """Vòng lặp chờ sóng vệ tinh đạt mức tối thiểu trước khi gửi."""
        self.log(f"⏳ Đang chờ sóng vệ tinh đạt tối thiểu {min_bars}/5 vạch...")
        for i in range(1, max_attempts + 1):
            bars = self.check_signal()
            if bars >= min_bars:
                self.log(f"🎉 Đã bắt được sóng vệ tinh đủ mạnh ({bars}/5 vạch)!")
                return True
            if i < max_attempts:
                self.log(f"   [Lần {i}/{max_attempts}] Sóng chưa đủ. Đợi {interval}s (Lưu ý: Hướng anten thẳng lên trời)..." )
                time.sleep(interval)
        self.log(f"❌ Sau {max_attempts} lần kiểm tra vẫn chưa có đủ sóng vệ tinh.")
        return False

    def send_message(self, message_text, target_serial="0235708", wait_sbd=20):
        """
        Nạp tin nhắn và truyền qua vệ tinh Iridium tới target_serial.
        Theo chuẩn RockBLOCK-to-RockBLOCK Direct Addressing:
        - Tiền tố: 'RB' + <Serial 7 chữ số của thiết bị nhận>
        - Ví dụ gửi tới 0235708: 'RB0235708Hello'
        - Gateway Rock Seven tự động đọc tiền tố, chuyển đến RockBLOCK 0235708
          và cắt bỏ tiền tố 'RB0235708' trước khi trao cho bên nhận.
        """
        # Chuẩn hóa target_serial thành đúng 7 chữ số (bù số 0 ở đầu nếu thiếu)
        if target_serial:
            clean_serial = f"{int(target_serial):07d}"
            full_payload = f"RB{clean_serial}{message_text}"
            prefix_info = f"RB{clean_serial}"
        else:
            full_payload = message_text
            clean_serial = "None (Broadcast/Portal default)"
            prefix_info = "None"

        self.log("=" * 60)
        self.log(f"📤 BẮT ĐẦU QUÁ TRÌNH GỬI THÔNG ĐIỆP ĐẾN ROCKBLOCK {clean_serial}:")
        self.log(f"🎯 Đích đến (Target Serial): {clean_serial}")
        self.log(f"🏷️  Tiền tố Direct Addressing: {prefix_info}")
        self.log(f"📝 Nội dung thực tế        : \"{message_text}\"")
        self.log(f"📦 Toàn bộ chuỗi gửi đi (AT): \"{full_payload}\"")
        self.log(f"📏 Tổng độ dài nạp vào modem: {len(full_payload.encode('ascii'))} bytes")
        self.log("=" * 60)

        # Bước 1: Xóa bộ đệm MO cũ để tránh gửi sót dữ liệu trước
        self.send_cmd("AT+SBDD0", wait=1)

        # Bước 2: Nạp chuỗi có tiền tố RB vào MO Buffer qua lệnh AT+SBDWT
        write_cmd = f"AT+SBDWT={full_payload}"
        write_resp = self.send_cmd(write_cmd, wait=1)
        if "OK" not in write_resp:
            self.log(f"❌ Không thể nạp tin nhắn vào bộ đệm! Phản hồi: {write_resp}")
            return False
        self.log("✅ Đã nạp thông điệp vào bộ đệm MO thành công.")

        # Bước 3: Kích hoạt phiên truyền dữ liệu vệ tinh SBD (AT+SBDIX)
        self.log("🚀 Đang gọi AT+SBDIX (Đang kết nối vệ tinh Iridium, vui lòng đợi 10-25 giây)..." )
        sbdix_resp = self.send_cmd("AT+SBDIX", wait=wait_sbd)
        self.log(f"📥 Phản hồi từ phiên SBDIX:\n{sbdix_resp}")

        # Bước 4: Phân tích kết quả trả về từ +SBDIX
        if "+SBDIX:" in sbdix_resp:
            try:
                raw_params = sbdix_resp.split("+SBDIX:")[1].strip().split(",")
                mo_status = int(raw_params[0].strip())
                momsn = int(raw_params[1].strip())
                mt_status = int(raw_params[2].strip())
                mt_len = int(raw_params[4].strip())

                desc = self.MO_STATUS_DESC.get(mo_status, f"Mã trạng thái MO không xác định: {mo_status}")
                self.log(f"📊 Kết quả MO Status = {mo_status}: {desc}")
                self.log(f"🔢 MOMSN (Số thứ tự tin nhắn gửi): {momsn}")
                
                # Kiểm tra nếu đồng thời có tin nhắn MT gửi về
                if mt_status == 1:
                    self.log(f"📬 [Thông báo kèm theo] Modem nhận được 1 tin nhắn MT ({mt_len} bytes) từ Gateway!")

                # Bước 5: Xóa bộ đệm MO sau khi gửi
                self.send_cmd("AT+SBDD0", wait=1)

                if mo_status in [0, 1, 2]:
                    self.log("✨ KẾT QUẢ: GỬI TIN NHẮN QUA VỆ TINH THÀNH CÔNG RỰC RỠ!")
                    self.log(f"ℹ️  Gateway Rock Seven sẽ bóc bỏ 'RB{clean_serial}' và chuyển nội dung '\"{message_text}\"' đến RockBLOCK {clean_serial}.")
                    return True
                else:
                    self.log(f"⚠️ GỬI THẤT BẠI: Mã lỗi MO = {mo_status}. (Gợi ý: Cần anten ngoài trời, góc quét bầu trời thoáng)")
                    return False
            except Exception as ex:
                self.log(f"❌ Lỗi khi phân tích chuỗi phản hồi SBDIX: {ex}")
                return False
        else:
            self.log("❌ Modem không trả về chuỗi kết quả +SBDIX. Vui lòng kiểm tra lại cáp và nguồn cấp!")
            return False


In [ ]:
# CELL 3: KHỞI TẠO ĐỐI TƯỢNG VÀ KẾT NỐI VỚI ROCKBLOCK SENDER
sender = RockBlockSender(port=PORT, baudrate=BAUDRATE, log_file=LOG_FILE)

# Mở cổng kết nối
if sender.connect():
    # Cấu hình modem và đọc IMEI
    ready = sender.init_modem()
    if not ready:
        print("⚠️ Khởi tạo modem thất bại. Vui lòng kiểm tra cổng COM hoặc rút cắm lại thiết bị.")
else:
    print("❌ Không mở được cổng Serial. Hãy kiểm tra lại biến PORT ở Cell 1!")


In [ ]:
# CELL 4: KIỂM TRA SÓNG VỆ TINH TRƯỚC KHI GỬI
# Lưu ý: Mạng Iridium hoạt động tốt nhất khi anten được đặt ngoài trời hoặc sát cửa sổ hướng lên trời.
print("📡 Bắt đầu kiểm tra sóng Iridium...")
has_signal = sender.wait_for_signal(min_bars=2, max_attempts=6, interval=5)

if has_signal:
    print("✅ Sóng vệ tinh sẵn sàng cho việc phát tin.")
else:
    print("⚠️ Cảnh báo: Sóng vệ tinh yếu hoặc chưa bắt được. Nếu gửi có thể gặp lỗi MO=32 (Timeout).")


In [ ]:
# CELL 5: THỰC HIỆN GỬI THÔNG ĐIỆP SANG ROCKBLOCK NHẬN (TARGET: 0235708)
# Nhập nội dung tin nhắn bạn muốn gửi ở biến custom_message dưới đây:
custom_message = f"HELLO FROM 0235707 TO 0235708 TIME={datetime.now().strftime('%H:%M:%S')}"

# Hàm send_message sẽ tự động ghép tiền tố 'RB0235708' vào đầu lệnh AT+SBDWT:
# Ví dụ: AT+SBDWT=RB0235708HELLO FROM 0235707...
success = sender.send_message(
    message_text=custom_message,
    target_serial=TARGET_RECEIVER_SERIAL,
    wait_sbd=20
)

if success:
    print("\n" + "=" * 60)
    print(f"🎉 THÀNH CÔNG: Đã gửi thông điệp tới RockBLOCK {TARGET_RECEIVER_SERIAL}!")
    print(f"💡 Lưu ý: Khi RockBLOCK {TARGET_RECEIVER_SERIAL} đọc tin nhắn bằng AT+SBDRT,")
    print(f"   nó sẽ nhận được nội dung nguyên bản: '{custom_message}'")
    print("=" * 60)
else:
    print("\n❌ Gửi tin nhắn thất bại. Hãy kiểm tra anten hướng lên bầu trời thoáng đãng!")


In [ ]:
# CELL 6: VÍ DỤ GỬI DỮ LIỆU ĐỊNH VỊ HOẶC CẢM BIẾN (TELEMETRY) TỚI RECEIVER 0235708
telemetry_data = {
    "lat": 21.0285,
    "lon": 105.8542,
    "temp": 28.5,
    "bat": 98
}

# Chuyển đổi thành chuỗi ngắn gọn dạng CSV để tiết kiệm byte truyền qua vệ tinh
compact_msg = f"GPS:{telemetry_data['lat']},{telemetry_data['lon']},T:{telemetry_data['temp']},B:{telemetry_data['bat']}"

print(f"📦 Dữ liệu nén gọn ({len(compact_msg)} bytes): {compact_msg}")
print(f"🎯 Chuỗi đầy đủ khi gửi sẽ là: RB{int(TARGET_RECEIVER_SERIAL):07d}{compact_msg}")

# Bỏ chú thích dòng dưới để gửi dữ liệu telemetry khi muốn:
# sender.send_message(compact_msg, target_serial=TARGET_RECEIVER_SERIAL, wait_sbd=20)


In [ ]:
# CELL 7: ĐÓNG KẾT NỐI KHI HOÀN TẤT
sender.disconnect()
print(f"📄 Toàn bộ tiến trình đã được lưu lại tại file: {LOG_FILE}")
